# Notebook 02 — VIX Preprocessing

Handles: VIX Cleaning, Date Handling, Previous-Day VIX Alignment, sigma Creation

**Key rule:** sigma = previous trading day's VIX close (NOT same-day VIX)

**Input :** `data/raw/vix/vix_daily.csv`  
**Output:** `data/processed/intermediate/vix_cleaned.csv`

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'src', 'preprocessing'))
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
print('Libraries loaded.')

## Step 1 — Run the VIX Preprocessing Module

In [ ]:
import vix_preprocessing

df_vix = vix_preprocessing.run(verbose=True)


## Step 2 — Inspect the Cleaned VIX Dataset

In [ ]:
print('Shape:', df_vix.shape)
df_vix.head(10)


## Step 3 — Temporal Alignment Verification

For each option trading date D, `sigma` is the VIX close from the **previous** trading day.

The implementation uses `VIX_close.shift(1)` on date-sorted data, which correctly assigns
the prior trading day's VIX to each row without any future information leakage.

In [ ]:
# Show the sigma alignment for the option date (2023-08-25)
option_date = pd.Timestamp('2023-08-25')
mask = (df_vix['date'] >= option_date - pd.Timedelta(days=5)) & \
       (df_vix['date'] <= option_date)
print('VIX values around option trading date:')
print(df_vix[mask][['date', 'VIX_close', 'sigma']].to_string(index=False))
sigma_used = df_vix.loc[df_vix['date'] == option_date, 'sigma'].values
print(f'\nsigma used for options on {option_date.date()}: {sigma_used}')

## Step 4 — VIX Time Series Visualisation

In [ ]:
# Plot last 2 years of VIX
df_recent = df_vix[df_vix['date'] >= '2021-01-01'].copy()
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df_recent['date'], df_recent['VIX_close'], color='steelblue', linewidth=1)
ax.axvline(pd.Timestamp('2023-08-25'), color='red', linestyle='--', label='Option date')
ax.set_title('VIX Daily Close (2021-2024)')
ax.set_xlabel('Date')
ax.set_ylabel('VIX Level')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join('..', 'outputs', 'figures', '02_vix_timeseries.png'), dpi=100)
plt.show()
print('Plot saved.')

## Summary

- VIX raw dataset: 9,266 trading days (1990-01-02 to 2026)
- `sigma` = previous trading day's VIX close (via `.shift(1)` on sorted data)
- Output saved to `data/processed/intermediate/vix_cleaned.csv`
- Columns: `date`, `VIX_close`, `sigma`